In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df=pd.read_csv("student_performance_dataset.csv")

In [3]:
df.shape

(1000, 12)

In [4]:
df.head()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  1000 non-null   int64  
 1   gender                      1000 non-null   object 
 2   study_time_hours            1000 non-null   float64
 3   attendance_percent          1000 non-null   float64
 4   sleep_hours                 1000 non-null   float64
 5   parental_education          898 non-null    object 
 6   internet_access             1000 non-null   object 
 7   extracurricular_activities  1000 non-null   object 
 8   part_time_job               1000 non-null   object 
 9   previous_grade              1000 non-null   float64
 10  final_exam_score            1000 non-null   float64
 11  final_grade                 1000 non-null   object 
dtypes: float64(5), int64(1), object(6)
memory usage: 93.9+ KB


In [6]:
df.isnull().sum()

student_id                      0
gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
dtype: int64

In [7]:
df=df.drop(["student_id","sleep_hours","attendance_percent","internet_access","previous_grade"],axis=1)

In [8]:
df["parental_education"].value_counts()

parental_education
High School    356
Bachelors      308
Masters        184
PhD             50
Name: count, dtype: int64

In [9]:
df.shape

(1000, 7)

In [10]:
x_train,x_test,y_train,y_test=train_test_split(df.drop('part_time_job',axis=1),
                                                       df['part_time_job'],test_size=0.2,random_state=42)

In [11]:
x_train

,gender,study_time_hours,parental_education,extracurricular_activities,final_exam_score,final_grade
29,Female,2.4,Bachelors,Yes,89.8,B
535,Female,2.0,High School,No,77.6,C
695,Female,3.0,High School,Yes,77.5,C
557,Female,3.4,High School,Yes,73.1,C
836,Male,4.5,Bachelors,Yes,100.0,A
...,...,...,...,...,...,...
106,Female,3.6,High School,Yes,75.7,C
270,Female,3.5,Masters,Yes,75.1,C
860,Female,3.6,High School,No,91.1,A
435,Female,5.6,High School,Yes,93.4,A


In [12]:

# Pipeline: impute missing values THEN ordinally encode parental_education
# Pehle saare missing values fill → phir saare values encode.

edu_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OrdinalEncoder(categories=[['High School', 'Bachelors', 'Masters', 'PhD']]))
])

transformer = ColumnTransformer(
    transformers=[ 
        ('tnf1', edu_pipeline, ['parental_education']),# missing value fill → ordinal encoding transform in the column parental_education
        ('tnf2', OrdinalEncoder(categories=[['A', 'B', 'C', 'D', 'F']]), ['final_grade']),
        ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'extracurricular_activities'])
    ],
    remainder='passthrough'
)

In [13]:
x_train_new = transformer.fit_transform(x_train)

In [14]:
x_test_new=transformer.transform(x_test)

In [15]:
x_train_new = pd.DataFrame(
    x_train_new,
    columns=transformer.get_feature_names_out()
)

In [16]:
x_train_new

,tnf1__parental_education,tnf2__final_grade,tnf3__gender_Male,tnf3__extracurricular_activities_Yes,remainder__study_time_hours,remainder__final_exam_score
0,1.0,1.0,0.0,1.0,2.4,89.8
1,0.0,2.0,0.0,0.0,2.0,77.6
2,0.0,2.0,0.0,1.0,3.0,77.5
3,0.0,2.0,0.0,1.0,3.4,73.1
4,1.0,0.0,1.0,1.0,4.5,100.0
...,...,...,...,...,...,...
795,0.0,2.0,0.0,1.0,3.6,75.7
796,2.0,2.0,0.0,1.0,3.5,75.1
797,0.0,0.0,0.0,0.0,3.6,91.1
798,0.0,0.0,0.0,1.0,5.6,93.4
